In [1]:
import os
from getpass import getpass
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import random
from tqdm import tqdm

from transformers import AutoTokenizer, AutoModel

from sklearn.model_selection import train_test_split

SEED = 42


In [2]:
# Check to see if files are downloaded before running the following cell
!ls -la data

ls: cannot access 'data': No such file or directory


In [3]:
# Get Kaggle API Token
os.environ['KAGGLE_API_TOKEN'] = getpass("Kaggle API token: ")

# Download kaggle competition datasets
!kaggle competitions download -c llm-classification-finetuning

# Unzip the files
!unzip -q llm-classification-finetuning.zip -d data

# Lists the files
!ls -la data

100% 57.0M/57.0M [00:00<00:00, 170MB/s] 

total 179884
drwxr-xr-x 2 root root      4096 Jul 24 21:21 .
drwxr-xr-x 1 root root      4096 Jul 24 21:21 ..
-rw-r--r-- 1 root root       237 Oct  9  2024 sample_submission.csv
-rw-r--r-- 1 root root     10658 Oct  9  2024 test.csv
-rw-r--r-- 1 root root 184175071 Oct  9  2024 train.csv


# Archetecure of model:

### input variables
```
prompt = prompt
response_1 = either reponse_a or response_b
response_2 = the other response_a/b
```
### Encoding model
```
model_1 = DeBERTA v3 base
```

### Classification model
```
model_2 = classification model
```
### pipeline of data
```
prompt -> model_1 -> output_p
response_1 -> model_1 -> output_1
response_2 -> model_1 -> output_2

(output_p, output_1, output_2) -> model_2 -> (prob of first response, prob of second response, prob of tie)
```



In [4]:
class ResponsePreferenceDataset(Dataset):
    """
    A custom Dataset class for handling prompts, responses from two models, and preference labels.
    This Dataset uses lazy loading.
    """
    def __init__(self, prompts, responses_a, responses_b, labels, tokenizer, max_len=512, prob=0.5):
        """
        Initializes the dataset.

        Args:
            prompts (list): A list of strings containing the prompts.
            responses_a (list): A list of strings containing the responses to
                model_a.
            responses_b (list): A list of strings containing the responses to
                model_b.
            labels (ndarray): A 1-d array of integers containing the preferred
                response.
                - 0: response_a preferred
                - 1: response_b preferred
                - 2: tie
            tokenizer (object): An object used to convert text into numerical
                indices.
            max_len (int): The maximum sequence length allowed for each
                prompt/response.
            prob (float): The probability (between 0 and 1) of permuting the
                order of responses.
        """
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.prob = prob
        self.prompts = prompts
        self.responses_a = responses_a
        self.responses_b = responses_b
        self.labels = labels

    def __len__(self):
        """
        Returns the number of samples in the dataset.

        Returns:
            int: The number of samples in the dataset.
        """
        return len(self.prompts)

    def __getitem__(self, idx):
        """
        Retrieves and tokenizes one sample, randomly swaps response_a/b (with
            probability self.prob) to reduce positional bias in training.

        Args:
            idx (int): Index of the sample to retrieve.

        Returns:
            dict: A dictionary containing:
                - prompt (Tensor): Token IDs for the prompt.
                - prompt_mask (Tensor): Attention mask for the prompt.
                - response_0 (Tensor): Token IDs for response_a (possibly
                    swapped).
                - response_0_mask (Tensor): Attention mask for response_0.
                - response_1 (Tensor): Token IDs for response_b (possibly
                    swapped).
                - response_1_mask (Tensor): Attention mask for response_1.
                - label (Tensor): Integer class label 0/1/2,
                    adjusted if swapped.
        """
        prompt = self.prompts[idx]
        label = self.labels[idx]
        response_a = self.responses_a[idx]
        response_b = self.responses_b[idx]

        # Swap responses and adjust label with probability self.prob.
        if random.random() < self.prob:
            response_a, response_b = response_b, response_a
            if label == 0:
                label = 1
            elif label ==1:
                label = 0

        # Switch from response_a/response_b to response_0/response_1 since
        # response_a and response_b may have been swapped.

        response_0, response_1 = response_a, response_b

        encoded_p = self.tokenizer(prompt, max_length=self.max_len,
                                   truncation=True, padding="max_length",
                                   return_tensors="pt")
        encoded_0 = self.tokenizer(response_0, max_length=self.max_len,
                                   truncation=True, padding="max_length",
                                   return_tensors="pt")
        encoded_1 = self.tokenizer(response_1, max_length=self.max_len,
                                   truncation=True, padding="max_length",
                                   return_tensors="pt")

        # After retrieving the tokens, we remove the extra dimension that was added
        # by the tokenizer (the batch dim) even for a single string.
        prompt = encoded_p["input_ids"].squeeze(0)
        response_0 = encoded_0["input_ids"].squeeze(0)
        response_1 = encoded_1["input_ids"].squeeze(0)

        prompt_mask = encoded_p["attention_mask"].squeeze(0)
        response_0_mask = encoded_0["attention_mask"].squeeze(0)
        response_1_mask = encoded_1["attention_mask"].squeeze(0)

        label = torch.tensor(label, dtype=torch.long)

        return {
            "prompt": prompt, "prompt_mask": prompt_mask,
            "response_0": response_0, "response_0_mask": response_0_mask,
            "response_1": response_1, "response_1_mask": response_1_mask,
            "label": label
        }




In [5]:

df = pd.read_csv("data/train.csv")
#df = df.head(200)

# Convert one hot to 0,1,2
labels_one_hot = df[["winner_model_a","winner_model_b","winner_tie"]]
labels = labels_one_hot.values.argmax(axis = 1)


In [6]:


tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-v3-base")



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model: reconstructing file:   0%|          |  0.00B / 2.46MB            

spm.model: downloading bytes:           |  0.00B            

In [7]:
train_df, val_df, train_labels, val_labels = train_test_split(
    df,
    labels,
    test_size=0.2,
    stratify=labels,
    random_state=SEED
)

val_df, test_df, val_labels, test_labels = train_test_split(
    val_df,
    val_labels,
    test_size=0.5,
    stratify=val_labels,
    random_state=SEED
)

In [8]:
train_dataset = ResponsePreferenceDataset(
    prompts = train_df["prompt"].tolist(),
    responses_a = train_df["response_a"].tolist(),
    responses_b = train_df["response_b"].tolist(),
    labels= train_labels,
    tokenizer = tokenizer
)
val_dataset = ResponsePreferenceDataset(
    prompts = val_df["prompt"].tolist(),
    responses_a = val_df["response_a"].tolist(),
    responses_b = val_df["response_b"].tolist(),
    labels= val_labels,
    tokenizer = tokenizer,
    prob=0
)
test_dataset = ResponsePreferenceDataset(
    prompts = test_df["prompt"].tolist(),
    responses_a = test_df["response_a"].tolist(),
    responses_b = test_df["response_b"].tolist(),
    labels= test_labels,
    tokenizer = tokenizer,
    prob=0
)

In [20]:
batch_size = 32
train_loader = DataLoader( train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
val_loader = DataLoader( val_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
test_loader = DataLoader( test_dataset, batch_size=batch_size, shuffle=True, num_workers=2)

In [10]:
class PreferenceClassifier(nn.Module):
    """
    Classification head that takes pooled encoder representations of the prompt,
    response_0, and response_1 and predicts which response is preferred:
        - 0 response_0 preferred
        - 1 response_1 preferred
        - 2 tie
    """
    def __init__(self, hidden_state=768, num_classes=3):
        """
        Initializes the classifier.

        Args:
            - hidden_state (int): The dimension of the encoding of each text.
            - num_classes (int): The number of classes.
        """
        super().__init__()
        self.fc1 = nn.Linear(hidden_state*3, hidden_state)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.1)
        self.fc2 = nn.Linear(hidden_state, num_classes)

    def forward(self, output_p, output_0, output_1):
        """
        Executes the forward pass of the model to generate classification
        logits.

        Args:
            - output_p (Tensor): Encoded prompt.
            - output_0 (Tensor): Encoded response_0.
            - output_1 (Tensor): Encoded response_1.

        Returns:
            Tensor: Classification logits.
        """
        combined = torch.cat([output_p, output_0, output_1], dim=1)
        x = self.fc1(combined)
        x = self.relu(x)
        x = self.dropout(x)
        logits = self.fc2(x)

        return logits


In [26]:
class PreferenceModel(nn.Module):
    """
    A model that takes the encoder model and the classifier
    model to create the full end-to-end model. The model first encodes prompt,
    response_0, response_1 through the encoder model. The three outputs are fed
    through the classifier model to predict the user preference:
        - 0: response_0 is preferred
        - 1: response_1 is preferred
        - 2: tie
    """
    def __init__(self, model_encoder, model_classifier):
        """
        Initializes the model with the model encoder and model classifier.

        Args:
            model_encoder (nn.Module): A pretrained encoder model
                (e.g. DebertaV2Model) that outputs last_hidden_state given
                input_ids and attention_mask.
            model_classifier (PreferenceClassifier): The classification head
                that takes encoder outputs and predicts preference.
        """
        super().__init__()
        self.model_encoder = model_encoder
        self.model_classifier = model_classifier

    def forward(self,prompt, prompt_mask, response_0, response_0_mask, response_1, response_1_mask):
        """
        Executes the forward pass of the model.

        Args:
            prompt (Tensor): The Input tensor containing token indices of
                prompt.
            prompt_mask (Tensor): Attention mask for the prompt.
            response_0 (Tensor): The Input tensor containing token indices of
                response 0.
            response_0_mask (Tensor): Attention mask for response 0.
            response_1 (Tensor): The Input tensor containing token indices of
                response 1.
            response_1_mask (Tensor): Attention mask for response 1.

        Returns:
            Tensor: Classifier logits.
        """
        output_p = self.model_encoder(input_ids=prompt, attention_mask=prompt_mask).last_hidden_state[:,0,:]
        output_0 = self.model_encoder(input_ids=response_0, attention_mask=response_0_mask).last_hidden_state[:,0,:]
        output_1 = self.model_encoder(input_ids=response_1, attention_mask=response_1_mask).last_hidden_state[:,0,:]

        logits = self.model_classifier(output_p, output_0, output_1)

        return logits



In [27]:
# Testing PreferenceModel:
model_encoder = AutoModel.from_pretrained("microsoft/deberta-v3-base", torch_dtype=torch.float32)
model_classifier = model_2 = PreferenceClassifier()
model = PreferenceModel(model_encoder,model_classifier)


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.classifier.bias        | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [19]:
batch = next(iter(val_loader))
batch = {k: v[:1] for k,v in batch.items()} # Testing a small batch size
model.eval()
with torch.no_grad():
    output = model(
        batch["prompt"], batch["prompt_mask"],
        batch["response_0"], batch["response_0_mask"],
        batch["response_1"], batch["response_1_mask"]
    )
print(output.shape)
print(output)

torch.Size([1, 3])
tensor([[0.0358, 0.0576, 0.3020]])


In [28]:
loss = nn.CrossEntropyLoss()
learning_rate = 0.001
optimizer = optim.AdamW(model.parameters(), lr=learning_rate)

In [21]:
device = torch.device('cuda' if torch.cuda.is_available() else "cpu")
model = model.to(device)
print(f"Evaluating untrained model using device: {device}")
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for batch in tqdm(val_loader):
        batch = {k: v.to(device) for k,v in batch.items()}
        output = model(
            batch["prompt"], batch["prompt_mask"],
            batch["response_0"], batch["response_0_mask"],
            batch["response_1"], batch["response_1_mask"]
        )
        predicted = torch.argmax(output, 1)
        total += batch["label"].size(0)
        correct += (predicted == batch["label"]).sum().item()
acc = 100 * correct / total
print(f"Intitial test accuracy before training: {acc}")

Evaluating untrained model using device: cuda


100%|██████████| 180/180 [13:00<00:00,  4.34s/it]

Intitial test accuracy before training: 30.880306193458594


### Intial run. Time with calling encoder 3 separate times:
100%|██████████| 180/180 [13:00<00:00,  4.34s/it]
Intitial test accuracy before training: 30.880306193458594

Note:
- adding workers didn't speed up time
- changed from calling encoder 3 times into one time by stacking into one big batch. Didn't improve by much (30 seconds per batch)
- Considered changing max_len on token lengths, but stats on data shows we would lose too much info:

prompt: mean=101.0, median=27,  95th=377, 99th=1390, max=9358
response_a: mean=347.0, median=264,  95th=963, 99th=1961, max=9557
response_b: mean=350.6, median=266,  95th=960, 99th=2014, max=15685

In [17]:
print(next(model.parameters()).device)
print(tokenizer.is_fast)

cuda:0
True


In [29]:
import numpy as np

prompt_lengths = [len(ids) for ids in tokenizer(train_df["prompt"].tolist(), truncation=False)["input_ids"]]
response_a_lengths = [len(ids) for ids in tokenizer(train_df["response_a"].tolist(), truncation=False)["input_ids"]]
response_b_lengths = [len(ids) for ids in tokenizer(train_df["response_b"].tolist(), truncation=False)["input_ids"]]

for name, lengths in [("prompt", prompt_lengths), ("response_a", response_a_lengths), ("response_b", response_b_lengths)]:
    lengths = np.array(lengths)
    print(f"{name}: mean={lengths.mean():.1f}, median={np.median(lengths):.0f},  95th={np.percentile(lengths, 95):.0f}, 99th={np.percentile(lengths,99):.0f}, max={lengths.max()}")

prompt: mean=101.0, median=27,  95th=377, 99th=1390, max=9358
response_a: mean=347.0, median=264,  95th=963, 99th=1961, max=9557
response_b: mean=350.6, median=266,  95th=960, 99th=2014, max=15685
